# פרויקט לדוגמה — Used Cars — חיזוי מחיר רכב משומש

**שאלת חקר:** האם ניתן להעריך מחיר רכב Audi משומש על פי נתוני הרכב?

**מקור:** מחברת Used-Cars שהועלתה. המחברת הזאת נבנתה מחדש, ללא העתקת זהויות תלמידים או פלטים. הפרדה נכונה בין מחיר (y) למאפיינים ואי שימוש בקטגוריית מחיר נגזרת.

**סדר:** שאלה → מקור ומילון נתונים → EDA → איכות נתונים → X/y → Train/Test → Pipeline → אימון → CV → מבחן סופי → מסקנות.

זוהי דוגמה פדגוגית שדורשת הרצה ואימות על נתוני המקור; אין להסתמך על תוצאות שלא הופקו בפועל.

## 1. טעינת מאגר ומקור הנתונים
[עמוד Dataset ב-Kaggle](https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
# התקנה/הורדה ב-Colab עם גישה לרשת. שם הקובץ נבדק לפי הנתיב, לא לפי ניחוש כתובת CSV.
%pip -q install kagglehub
import kagglehub
from pathlib import Path
folder=Path(kagglehub.dataset_download("adityadesai13/used-car-dataset-ford-and-mercedes"))
files=sorted(folder.rglob("*.csv"))
print("קובצי CSV זמינים:",[p.name for p in files])
match=[p for p in files if p.name.lower()=="audi.csv"]
if not match:
    raise FileNotFoundError("לא נמצא audi.csv. בדקו את שמות הקבצים והגרסה שהורדה.")
df=pd.read_csv(match[0])
print("Rows, columns:",df.shape)
display(df.head())

## 2. הבנת הנתונים
בדקו שמות עמודות, סוגי נתונים, חסרים וכפילויות. השלימו בתא Markdown את הגדרת יחידת התצפית, משמעות המטרה ומקור הנתונים.

In [ ]:
print(df.dtypes)
print("חסרים:\n",df.isna().sum().sort_values(ascending=False).head(15))
print("שורות זהות:",df.duplicated().sum())
print("עמודות:",list(df.columns))

## 3. בחירת מטרה ומאפיינים
**y = price**. מאפיינים המוצעים: model, year, transmission, mileage, fuelType, tax, mpg, engineSize. אין לכלול יעד נגזר או מאפיינים המכילים בפועל את תוצאות ההישגים.

In [ ]:
target="price"
requested=["model","year","transmission","mileage","fuelType","tax","mpg","engineSize"]
needed=[target]+requested
missing=[col for col in needed if col not in df.columns]
if missing:
    raise ValueError("חסרות עמודות שנדרשות בדוגמה: "+str(missing)+". אין להמשיך בלי להתאים את המילון למאגר המקורי.")
data=df[needed].dropna(subset=[target]).copy()
X=data[requested]
y=data[target]
print("מספר רשומות עם יעד:",len(data))
print("מאפיינים:",list(X.columns))

## 4. EDA — התפלגות יעד וסוגי משתנים
נסחו שתי שאלות חקר. כל גרף חייב כותרת, תיאור ממצא ומגבלה.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
sns.histplot(data=data,x=target,ax=ax[0],bins=25)
sns.scatterplot(data=data,x="mileage",y=target,alpha=.25,ax=ax[1])
ax[0].set_title("Price distribution");ax[1].set_title("Mileage vs price")
plt.tight_layout();plt.show()
print(data.select_dtypes(include="number").describe())

## 5. הכנה בלי דליפת מידע
נחלק את הנתונים **לפני** השלמה/תקנון/קידוד. Pipeline ילמד פרמטרים רק ב־Train ובקיפולי אימון של CV.

In [ ]:
numeric=X.select_dtypes(include="number").columns.tolist()
category=[c for c in X.columns if c not in numeric]
pre=ColumnTransformer([
    ("num",Pipeline([("fill",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric),
    ("cat",Pipeline([("fill",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),category)
])
print("numeric:",numeric,"categorical:",category)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
print("train:",X_train.shape,"test:",X_test.shape)

## 6. רגרסיה בסיסית ו־Cross Validation
נשתמש ב־LinearRegression; נשווה מדדי Test למדד CV מ־Train. חשוב לדווח על יחידות המחיר.

In [ ]:
model=Pipeline([("prepare",pre),("reg",LinearRegression())])
cv=KFold(n_splits=5,shuffle=True,random_state=42)
cv_scores=cross_val_score(model,X_train,y_train,cv=cv,scoring="r2")
print("CV R² mean:",cv_scores.mean())
model.fit(X_train,y_train)
pred=model.predict(X_test)
print("Test R²:",r2_score(y_test,pred))
print("Test MAE:",mean_absolute_error(y_test,pred))
plt.scatter(y_test,pred,alpha=.25)
plt.xlabel("Actual price");plt.ylabel("Predicted price");plt.title("Actual vs predicted")
plt.show()

## 7. מסקנות, מגבלות, אתיקה ורפלקציה
כתבו: (א) תשובה לשאלת החקר, (ב) שתי תובנות מבוססות גרפים, (ג) השוואה לבסיס או CV, (ד) מגבלת נתונים אחת, (ה) מדוע אין להסיק סיבתיות או להכליל למדגם אחר, (ו) מקורות ורישוי.

**הגשה:** Notebook שניתן להריץ; מילון משתנים; טבלת ניקוי; 3 גרפים עם פירוש; מדדים; תיעוד הפרדת Train/Test; סיכום מסודר.

## 8. בדיקת איכות
- האם נתוני CSV זמינים וגרסתם מתועדת?
- האם price אינה נכללת בין המאפיינים?
- האם כל המרות הנתונים נלמדות רק על Train?
- האם המדד מחושב ב־Test שלא שימש לבחירת מודל?
- האם המודל והנתונים מתאימים לדרישות המחוון התקף?
- האם יש אישור לשימוש בנתוני המקור ולהפצתם?